# CoT Consistency Experiment

**Hypothesis:** A model trained on a problem will reproduce a more consistent (less diverse) chain-of-thought across repeated runs than when solving a problem it has never seen.

**Design:**
- Each sample in `sample_30.jsonl` has a `source_dataset` field indicating where its contamination came from.
- We map each `source_dataset` to the model that was trained on that data.
- We generate `N_RUNS` completions per sample at `temperature=1.0` (high randomness).
- If the model has memorized the solution, runs should converge; if not, they should diverge.

| source_dataset | Model assigned |
|---|---|
| `openthoughts` | `open-thoughts/OpenThinker-7B` |
| `s1` | `simplescaling/s1.1-7B` |
| `tulu` | `allenai/Llama-3.1-Tulu-3-8B` |
| `none` (clean) | `Qwen/Qwen2.5-7B-Instruct` (neutral baseline) |

In [ ]:
# All packages are pre-installed in the project venv (.venv/).
# This cell is a no-op if you're using the "Data Decontamination" kernel (recommended).
# If you ever need to add a package, run from a terminal:
#   .venv/bin/pip install <package>
import sys
print(f"Python: {sys.executable}")

In [ ]:
import json, re, os, difflib, time
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

import os

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN or HF_TOKEN.startswith('hf_YOUR'):
    raise ValueError(
        "\nHF_TOKEN is not set.\n"
        "  1. Go to https://huggingface.co/settings/tokens\n"
        "  2. Create a token with 'Make calls to Inference Providers' enabled\n"
        "  3. export HF_TOKEN=hf_... then restart the kernel"
    )
print(f"HF_TOKEN: {HF_TOKEN[:8]}{'*' * (len(HF_TOKEN) - 8)}")

N_RUNS      = 5
MAX_TOKENS  = 1024
SAMPLE_FILE = "sample_30.jsonl"
RUNS_CACHE  = "cot_runs_cache.json"

# ── Model mapping ──────────────────────────────────────────────────────────────
# We use the closest publicly-hosted proxy for each research model.
# The ideal models (s1.1-7B, OpenThinker-7B, Tulu-3-8B) are not on HF serverless.
#
#   openthoughts → DeepSeek-R1-Distill-Qwen-7B
#       Both are reasoning models trained on synthetic CoT data with the same
#       Qwen2.5-7B base. OpenThoughts-114k and DeepSeek-R1's distillation data
#       heavily overlap (MATH, AMC, AIME style problems).
#
#   s1 → Qwen/Qwen2.5-7B-Instruct
#       s1.1-7B IS Qwen2.5-7B fine-tuned on the s1 dataset; using the base
#       instruction model is the closest available proxy.
#
#   tulu → meta-llama/Llama-3.1-8B-Instruct
#       Tulu-3-8B IS Llama-3.1-8B fine-tuned on Tulu data; base instruct
#       model is the closest available proxy.
#
#   none (clean) → Qwen/Qwen2.5-7B-Instruct
#       Neutral baseline — no special reasoning fine-tune.

DATASET_TO_MODEL = {
    "openthoughts": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "s1":           "Qwen/Qwen2.5-7B-Instruct",
    "tulu":         "meta-llama/Llama-3.1-8B-Instruct",
    "none":         "Qwen/Qwen2.5-7B-Instruct",
}
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print(f"\n  N_RUNS     : {N_RUNS}")
print(f"  MAX_TOKENS : {MAX_TOKENS}")
print("  Model map  :")
for k, v in DATASET_TO_MODEL.items():
    print(f"    {k:15s} → {v}")

In [ ]:
# ── Load samples ───────────────────────────────────────────────────────────────
samples = []
with open(SAMPLE_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

print(f"Loaded {len(samples)} samples\n")
print("Label distribution:")
for label, cnt in Counter(s['label'] for s in samples).items():
    print(f"  {label:15s}: {cnt}")
print()
print("Source dataset distribution:")
for src, cnt in Counter(s['source_dataset'] for s in samples).items():
    print(f"  {src:15s}: {cnt}")
print()
print("Contamination type distribution:")
for ct, cnt in Counter(s['contamination_type'] for s in samples).items():
    print(f"  {ct:15s}: {cnt}")

In [ ]:
# Preview the first few samples to understand the structure
preview_df = pd.DataFrame([
    {
        "sample_id":        s["sample_id"],
        "label":            s["label"],
        "contamination_type": s["contamination_type"],
        "source_dataset":   s["source_dataset"],
        "model_to_use":     DATASET_TO_MODEL.get(s["source_dataset"], FALLBACK_MODEL),
        "subject":          s["subject"],
        "level":            s["level"],
        "problem_snippet":  s["problem"][:60].replace("\n", " ") + "...",
    }
    for s in samples
])
preview_df

In [ ]:
# ── Generation function ────────────────────────────────────────────────────────

def generate_cot_runs(
    problem: str,
    model_id: str,
    num_runs: int = N_RUNS,
    max_tokens: int = MAX_TOKENS,
    retry_delay: float = 2.0,
) -> list[str]:
    """
    Call the HF Inference API `num_runs` times at temperature=1.0.
    - Immediately falls back to FALLBACK_MODEL on 'model not supported' errors.
    - Retries up to 3x on transient errors (rate limits, timeouts).
    """
    client = InferenceClient(model=model_id, token=HF_TOKEN)
    system_msg = (
        "You are a precise math assistant. "
        "Think step-by-step, show your reasoning, "
        "and end with your final answer inside \\boxed{}."
    )
    runs = []
    active_model = model_id

    for i in range(num_runs):
        for attempt in range(3):
            try:
                response = InferenceClient(model=active_model, token=HF_TOKEN).chat.completions.create(
                    messages=[
                        {"role": "system", "content": system_msg},
                        {"role": "user",   "content": problem},
                    ],
                    temperature=1.0,
                    max_tokens=max_tokens,
                )
                runs.append(response.choices[0].message.content.strip())
                break  # success — move to next run

            except Exception as e:
                err = str(e)
                # Hard failure: model isn't available — don't retry, swap immediately
                if "model_not_supported" in err or "not supported" in err.lower():
                    print(f"    ! {active_model} not on serverless → falling back to {FALLBACK_MODEL}")
                    active_model = FALLBACK_MODEL
                    break  # retry this run with new model on next attempt loop? No — restart run
                else:
                    print(f"    [run {i+1}, attempt {attempt+1}] {err[:120]}")
                    if attempt < 2:
                        time.sleep(retry_delay * (attempt + 1))

        # If we just switched model due to unsupported error, redo this run index
        # by not appending anything — the outer loop will retry naturally via
        # the fallback model on subsequent runs. Re-run this index explicitly:
        if active_model != model_id and len(runs) < i + 1:
            try:
                response = InferenceClient(model=active_model, token=HF_TOKEN).chat.completions.create(
                    messages=[
                        {"role": "system", "content": system_msg},
                        {"role": "user",   "content": problem},
                    ],
                    temperature=1.0,
                    max_tokens=max_tokens,
                )
                runs.append(response.choices[0].message.content.strip())
            except Exception as e:
                print(f"    ! Fallback also failed: {str(e)[:120]}")

    return runs

print("generate_cot_runs() defined.")

In [ ]:
# ── Smoke test: verify each model in the map is reachable ─────────────────────
# Run this cell first. It sends ONE short request per unique model.
# If a model fails, it will print the error and the model will auto-fallback
# to FALLBACK_MODEL during the main generation loop.

test_problem = "What is 7 × 8? Show your work briefly."
unique_models = list(dict.fromkeys(DATASET_TO_MODEL.values()))  # deduplicated, order-preserved

print(f"Testing {len(unique_models)} unique models with 1 run each ...\n")
for model_id in unique_models:
    print(f"  [{model_id}]")
    try:
        response = InferenceClient(model=model_id, token=HF_TOKEN).chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a math assistant."},
                {"role": "user",   "content": test_problem},
            ],
            temperature=0.0,
            max_tokens=64,
        )
        text = response.choices[0].message.content.strip().replace("\n", " ")
        print(f"    ✓  {text[:80]}")
    except Exception as e:
        err = str(e)
        if "model_not_supported" in err or "not supported" in err.lower():
            print(f"    ✗  NOT on serverless — will auto-fallback to {FALLBACK_MODEL}")
        elif "401" in err or "Unauthorized" in err:
            print(f"    ✗  Auth error — check your HF_TOKEN has 'Inference Providers' permission")
        else:
            print(f"    ✗  {err[:120]}")
    print()

In [ ]:
# ── Main generation loop ───────────────────────────────────────────────────────
# Checks RUNS_CACHE first — if you've already run this, it skips generation.
# Delete RUNS_CACHE or set FORCE_REGEN = True to re-run everything.

FORCE_REGEN = False

if os.path.exists(RUNS_CACHE) and not FORCE_REGEN:
    print(f"Loading cached runs from {RUNS_CACHE} ...")
    with open(RUNS_CACHE) as f:
        all_runs = json.load(f)
    print(f"Loaded {len(all_runs)} cached entries.")
else:
    print(f"Generating {N_RUNS} CoT runs per sample ({len(samples)} samples) ...")
    all_runs = []
    for sample in tqdm(samples, desc="Samples"):
        model_id = DATASET_TO_MODEL.get(sample["source_dataset"], FALLBACK_MODEL)
        print(f"\n  [{sample['sample_id']:02d}] {sample['label']:15s} | "
              f"{sample['source_dataset']:15s} | {model_id}")
        print(f"       Problem: {sample['problem'][:70].replace(chr(10), ' ')} ...")

        runs = generate_cot_runs(sample["problem"], model_id)
        print(f"       Got {len(runs)}/{N_RUNS} runs.")

        all_runs.append({
            "sample_id":          sample["sample_id"],
            "id":                  sample["id"],
            "problem":            sample["problem"],
            "answer":             sample["answer"],
            "label":              sample["label"],
            "contamination_type": sample["contamination_type"],
            "source_dataset":     sample["source_dataset"],
            "model_used":         model_id,
            "subject":            sample["subject"],
            "level":              sample["level"],
            "runs":               runs,
        })

    with open(RUNS_CACHE, "w") as f:
        json.dump(all_runs, f, indent=2)
    print(f"\nSaved to {RUNS_CACHE}")

In [ ]:
# ── Display CoT outputs ────────────────────────────────────────────────────────
# For a given sample_id, print all N runs side-by-side so you can visually
# inspect how much the reasoning varies.

def show_cot(sample_id: int, max_chars_per_run: int = 600):
    """Print the CoT runs for a specific sample."""
    entry = next((e for e in all_runs if e["sample_id"] == sample_id), None)
    if entry is None:
        print(f"sample_id {sample_id} not found")
        return

    print("=" * 80)
    print(f"Sample ID : {entry['sample_id']}")
    print(f"Problem   : {entry['problem'][:200].replace(chr(10), ' ')}")
    print(f"Answer    : {entry['answer']}")
    print(f"Label     : {entry['label']}  |  Type: {entry['contamination_type']}")
    print(f"Source    : {entry['source_dataset']}  |  Model: {entry['model_used']}")
    print("=" * 80)

    for i, run in enumerate(entry["runs"], 1):
        snippet = run[:max_chars_per_run].replace("\n", "↵ ")
        if len(run) > max_chars_per_run:
            snippet += " ..."
        print(f"\n── Run {i} ──")
        print(snippet)

# Show a contaminated sample and a clean sample
# (Change these IDs to inspect any sample you like)
show_cot(sample_id=1)   # contaminated (C_lex, source=s1)

In [ ]:
show_cot(sample_id=21)  # clean

In [ ]:
# ── Metric helper functions ────────────────────────────────────────────────────

def extract_answer(text: str) -> str:
    """Pull the last \\boxed{...} value, or the last number if none."""
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    if matches:
        return matches[-1].strip()
    nums = re.findall(r'-?\d+\.?\d*', text)
    return nums[-1] if nums else "UNKNOWN"


def answer_agreement(runs: list[str]) -> float:
    """
    Fraction of runs that produced the majority answer.
    1.0 = all runs agree; low value = diverse answers.
    """
    answers = [extract_answer(r) for r in runs]
    if not answers:
        return 0.0
    most_common_count = Counter(answers).most_common(1)[0][1]
    return most_common_count / len(answers)


def semantic_consistency(runs: list[str], encoder) -> float:
    """
    Mean pairwise cosine similarity of full CoT embeddings.
    Higher = more similar reasoning paths.
    """
    if len(runs) < 2:
        return 0.0
    embs = encoder.encode(runs, normalize_embeddings=True)
    sim_matrix = cosine_similarity(embs)
    n = len(runs)
    # upper triangle (excluding diagonal)
    total = np.sum(np.triu(sim_matrix, k=1))
    pairs = n * (n - 1) / 2
    return float(total / pairs)


def rouge_l_overlap(runs: list[str]) -> float:
    """
    Mean pairwise ROUGE-L F1 across all run pairs.
    Captures longer shared subsequences (structural similarity).
    """
    if len(runs) < 2:
        return 0.0
    scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            scores.append(scorer.score(runs[i], runs[j])['rougeL'].fmeasure)
    return float(np.mean(scores))


def sequence_overlap(runs: list[str]) -> float:
    """
    Mean pairwise SequenceMatcher ratio (character-level).
    Sensitive to verbatim recitation.
    """
    if len(runs) < 2:
        return 0.0
    ratios = []
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            ratios.append(difflib.SequenceMatcher(None, runs[i], runs[j]).ratio())
    return float(np.mean(ratios))


print("Metric functions defined.")

In [ ]:
# ── Load encoder for semantic consistency ─────────────────────────────────────
# Runs on CPU (will be slower than GPU but works locally).
# Uses all-mpnet-base-v2 — a strong general sentence encoder.
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Encoding on: {device}")
encoder = SentenceTransformer('all-mpnet-base-v2', device=device)
print("Encoder loaded.")

In [ ]:
# ── Compute all metrics ────────────────────────────────────────────────────────
print("Computing consistency metrics for all samples ...")
rows = []
for entry in tqdm(all_runs, desc="Metrics"):
    runs = entry["runs"]
    if len(runs) < 2:
        print(f"  Skipping sample {entry['sample_id']}: only {len(runs)} run(s).")
        continue

    # Extract individual answers for display
    predicted_answers = [extract_answer(r) for r in runs]

    rows.append({
        "sample_id":           entry["sample_id"],
        "label":               entry["label"],
        "contamination_type":  entry["contamination_type"],
        "source_dataset":      entry["source_dataset"],
        "model_used":          entry["model_used"].split("/")[-1],  # short name
        "subject":             entry["subject"],
        "level":               entry["level"],
        "true_answer":         entry["answer"],
        "predicted_answers":   ", ".join(predicted_answers),
        "answer_agreement":    answer_agreement(runs),
        "semantic_consistency": semantic_consistency(runs, encoder),
        "rouge_l_overlap":     rouge_l_overlap(runs),
        "sequence_overlap":    sequence_overlap(runs),
        "num_runs":            len(runs),
    })

results_df = pd.DataFrame(rows)
print(f"\nComputed metrics for {len(results_df)} samples.")
results_df

In [ ]:
# ── Save results ───────────────────────────────────────────────────────────────
results_df.to_csv("cot_consistency_results.csv", index=False)
print("Results saved to cot_consistency_results.csv")

In [ ]:
# ── Aggregate comparison: contaminated vs. clean ──────────────────────────────
# This is the core test of our hypothesis.
# We expect contaminated rows to have higher consistency scores.

metric_cols = ["answer_agreement", "semantic_consistency", "rouge_l_overlap", "sequence_overlap"]

agg = (
    results_df
    .groupby("label")[metric_cols]
    .agg(["mean", "std"])
    .round(3)
)
print("=== Mean consistency by label (contaminated vs clean) ===")
print(agg)
print()
print("Interpretation:")
print("  Higher values = more consistent CoT across runs = more memorization signal")

In [ ]:
# ── Breakdown by contamination type ───────────────────────────────────────────
# C_lex  = lexical contamination (near-identical wording)
# C_sem  = semantic contamination (same problem, different wording)
# clean  = not in any training set

type_agg = (
    results_df
    .groupby("contamination_type")[metric_cols]
    .mean()
    .round(3)
    .sort_values("answer_agreement", ascending=False)
)
print("=== Mean consistency by contamination type ===")
print(type_agg)

In [ ]:
# ── Breakdown by source dataset ────────────────────────────────────────────────
# Shows which training corpus produces the strongest memorization signal.

src_agg = (
    results_df
    .groupby(["source_dataset", "model_used"])[metric_cols]
    .mean()
    .round(3)
    .sort_values("answer_agreement", ascending=False)
)
print("=== Mean consistency by source dataset / model ===")
print(src_agg)

In [ ]:
# ── Per-sample ranked view ─────────────────────────────────────────────────────
# Sort all samples by answer_agreement descending.
# Contaminated samples should cluster at the top if the hypothesis holds.

ranked = results_df[
    ["sample_id", "label", "contamination_type", "source_dataset",
     "true_answer", "predicted_answers",
     "answer_agreement", "semantic_consistency", "rouge_l_overlap", "sequence_overlap"]
].sort_values("answer_agreement", ascending=False).reset_index(drop=True)

print("=== All samples ranked by answer agreement ===")
ranked

In [ ]:
# ── Optional: inspect CoTs for any sample ─────────────────────────────────────
# Useful for qualitative inspection — does the high-consistency run
# look like recitation vs genuine reasoning?

# Change sample_id to any value from the ranked table above
show_cot(sample_id=7)   # openthoughts contaminated (C_lex, 348 shared n-grams)

In [ ]:
show_cot(sample_id=25)  # clean